# Python: File Handling, Error Handling & Logging

### A 2-hour hands-on session (Basic ➜ Medium level)

**Instructor note:** Run every cell in order — later cells (especially in
Section 2 and the Mini Project) depend on files created earlier.

**Agenda**

1. File Handling — `open()`, modes, reading/writing, `with`, CSV, JSON
2. Error & Exception Handling — `try/except/else/finally`, common exceptions, `raise`, custom exceptions
3. Logging Basics — why logging beats `print()`, levels, console + file logging
4. Mini Project — putting it all together

---


## 1. File Handling

Almost every real program needs to read configuration, load data, or save
results to disk. Python's built-in `open()` function is the entry point for
all of this.

### 1.1 `open()` and file modes

| Mode | Meaning            | If file doesn't exist       | If file exists              |
| ---- | ------------------ | --------------------------- | --------------------------- |
| `r`  | Read (default)     | Error (`FileNotFoundError`) | Read from start             |
| `w`  | Write              | Creates it                  | **Overwrites/erases it**    |
| `a`  | Append             | Creates it                  | Writes at the end           |
| `r+` | Read **and** write | Error                       | Read/write, cursor at start |

Let's see each one in action.


In [9]:
# --- Simple example: WRITE mode ('w') ---
# 'w' mode creates the file if it doesn't exist,
# and OVERWRITES it completely if it does.

f = open("notes.txt", "w")      # open file for writing
f.write("Hello, this is line 1.\n")   # write() does NOT add \n automatically -> we add it ourselves
f.write("This is line 2.\n")
f.close()                        # IMPORTANT: always close a file you open manually

print("notes.txt has been created and written to.")


notes.txt has been created and written to.


In [10]:
# --- Simple example: READ mode ('r') ---
f = open("notes.txt", "r")   # open file for reading (this is the default mode)
content = f.read()           # read() returns the ENTIRE file as one string
f.close()

print("File content:")
print(content)


File content:
Hello, this is line 1.
This is line 2.



**Watch out:** if you open `notes.txt` again with `"w"`, everything above
will be erased before the new content is written. This is the #1 beginner
mistake with file handling.


In [11]:
# --- APPEND mode ('a') ---
# 'a' does NOT erase existing content -> it writes at the END of the file.

f = open("notes.txt", "a")
f.write("This line was appended, old content is safe.\n")
f.close()

# Let's confirm by reading it again
f = open("notes.txt", "r")
print(f.read())
f.close()


Hello, this is line 1.
This is line 2.
This line was appended, old content is safe.



In [4]:
f = open("notes2.txt", "r+")
first_line = f.readline()          # read just the first line, cursor moves forward
print("First line was:", first_line.strip())

FileNotFoundError: [Errno 2] No such file or directory: 'notes2.txt'

In [5]:
f = open("todo.txt", "r+")
f.write("File Handling/nOther Topics")
f.close()
 

FileNotFoundError: [Errno 2] No such file or directory: 'todo.txt'

In [12]:
# --- READ + WRITE mode ('r+') ---
# 'r+' lets you both read and write, but it does NOT erase the file
# and the "cursor" (write position) starts at the very beginning.

f = open("notes.txt", "r+")
first_line = f.readline()          # read just the first line, cursor moves forward
print("First line was:", first_line.strip())


f.write(">>> INSERTED TEXT <<<")   # this OVERWRITES characters right after the cursor
f.close()

f = open("notes.txt", "r")
print("\nFile after r+ write:")
print(f.read())
f.close()


First line was: Hello, this is line 1.

File after r+ write:
Hello, this is line 1.
This is line 2.
This line was appended, old content is safe.
>>> INSERTED TEXT <<<


Notice how `r+` **overwrote** part of line 2 instead of neatly inserting a
new line — this is because text files don't support "insert", only
"overwrite at the cursor position". This is exactly the kind of subtle bug
that trips people up, which is one reason we're careful with file modes.

### 1.2 Reading files: `read()`, `readline()`, `readlines()`

We already saw `read()` (whole file as one string). Let's compare the other two.


In [13]:
# Recreate a clean file for this demo
f = open("notes.txt", "w")
f.write("apple\nbanana\ncherry\n")
f.close()

# readline() -> reads ONE line at a time (including the trailing \n)
f = open("notes.txt", "r")
line1 = f.readline()
line2 = f.readline()
print("Line 1:", repr(line1))   # repr() lets us SEE the \n character
print("Line 2:", repr(line2))
f.close()


Line 1: 'apple\n'
Line 2: 'banana\n'


In [14]:
# readlines() -> reads ALL lines and returns them as a LIST of strings
f = open("notes.txt", "r")
all_lines = f.readlines()
f.close()

print(all_lines)
print("Number of lines:", len(all_lines))

# You can also loop over the file object directly (memory efficient for big files)
f = open("notes.txt", "r")
for line in f:
    print("Fruit:", line.strip())   # .strip() removes the trailing \n
f.close()


['apple\n', 'banana\n', 'cherry\n']
Number of lines: 3
Fruit: apple
Fruit: banana
Fruit: cherry


### 1.3 The Context Manager: `with open(...) as f:`

Every example above needed a manual `f.close()`. If an error happens
_before_ `close()` is reached, the file can stay open/locked. The `with`
statement fixes this: it **automatically closes the file**, even if an
exception occurs inside the block.

```python
with open("file.txt", "r") as f:
    data = f.read()
# file is CLOSED automatically here, no matter what happened above
```

This is the **standard, professional way** to work with files in Python —
use it from now on.


In [15]:
# Same read, but the safe/professional way
with open("notes.txt", "r") as f:
    data = f.read()
# no need to call f.close() !

print(data)
print("Is file closed now?", f.closed)   # True - 'with' closed it automatically


apple
banana
cherry

Is file closed now? True


In [16]:
# Medium example: writelines() + cleaning data on read
fruits = ["apple\n", "banana\n", "cherry\n", "date\n"]

with open("fruits.txt", "w") as f:
    f.writelines(fruits)          # writelines() writes a LIST of strings, no auto \n added

with open("fruits.txt", "r") as f:
    # list comprehension: read all lines and strip the \n from each
    cleaned = [line.strip() for line in f.readlines()]

print(cleaned)


['apple', 'banana', 'cherry', 'date']


### 1.4 Working with CSV files (`csv` module)

CSV (Comma-Separated Values) is the most common format for tabular data.
Technically you _could_ parse it yourself with `.split(",")`, but that
breaks the moment a value contains a comma or quotes — always use the
built-in `csv` module instead.


In [17]:
import csv

# --- Simple example: writing a CSV file with csv.writer ---
with open("students.csv", "w", newline="") as f:
    # newline="" prevents extra blank lines on Windows - always include it when writing CSV
    writer = csv.writer(f)
    writer.writerow(["name", "age", "marks"])     # header row
    writer.writerow(["Asha", 21, 88])
    writer.writerow(["Bibek", 22, 76])
    writer.writerow(["Chandra", 20, 91])

print("students.csv created.")


students.csv created.


In [18]:
# --- Reading it back with csv.reader ---
with open("students.csv", "r", newline="") as f:
    reader = csv.reader(f)
    for row in reader:
        print(row)     # each row is a plain LIST of strings, e.g. ['Asha', '21', '88']


['name', 'age', 'marks']
['Asha', '21', '88']
['Bibek', '22', '76']
['Chandra', '20', '91']


In [19]:
# --- Medium example: DictWriter / DictReader ---
# DictWriter/DictReader let you work with rows as DICTIONARIES (by column name)
# instead of by position - much easier to read and less error-prone.

fieldnames = ["name", "age", "marks"]
records = [
    {"name": "Asha",    "age": 21, "marks": 88},
    {"name": "Bibek",   "age": 22, "marks": 76},
    {"name": "Chandra", "age": 20, "marks": 91},
]

with open("students_dict.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()          # writes the header row from fieldnames
    writer.writerows(records)     # writes all dict rows at once

# Now read it back and compute the average marks -> a small real task
with open("students_dict.csv", "r", newline="") as f:
    reader = csv.DictReader(f)
    total = 0
    count = 0
    for row in reader:
        print(row)                       # row is a dict: {'name': 'Asha', 'age': '21', 'marks': '88'}
        total += int(row["marks"])       # CSV values are always STRINGS -> convert manually
        count += 1

print(f"\nAverage marks: {total / count:.2f}")


{'name': 'Asha', 'age': '21', 'marks': '88'}
{'name': 'Bibek', 'age': '22', 'marks': '76'}
{'name': 'Chandra', 'age': '20', 'marks': '91'}

Average marks: 85.00


### 1.5 Working with JSON files (`json` module)

JSON (JavaScript Object Notation) is the standard format for structured /
nested data — think dictionaries and lists, not flat tables. It's the most
common format used by web APIs and config files.

Key functions:
| Function | Purpose |
|------------------|-------------------------------------------|
| `json.dumps(obj)` | Python object ➜ JSON **string** |
| `json.dump(obj,f)`| Python object ➜ JSON **file** |
| `json.loads(s)` | JSON string ➜ Python object |
| `json.load(f)` | JSON file ➜ Python object |


In [1]:
import json

# --- Simple example: dict <-> JSON string ---
person = {"name": "Asha", "age": 21, "is_student": True, "skills": ["Python", "SQL"]}

json_string = json.dumps(person, indent=2)   # indent=2 makes it human-readable ("pretty print")
print(json_string)
print(type(json_string))   # it's just a str


{
  "name": "Asha",
  "age": 21,
  "is_student": true,
  "skills": [
    "Python",
    "SQL"
  ]
}
<class 'str'>


In [3]:
# --- Writing a dict directly to a JSON FILE ---
import json
with open("person.json", "w") as f:
    json.dump(person, f, indent=2)   # note: json.dump (no 's') writes straight to the file object

print("person.json created.")


person.json created.


In [4]:
# --- Reading a JSON file back into Python ---
with open("person.json", "r") as f:
    loaded_person = json.load(f)     # json.load (no 's') reads from a file object

print(loaded_person)
print(type(loaded_person))           # it's a normal Python dict again
print("Name:", loaded_person["name"])


{'name': 'Asha', 'age': 21, 'is_student': True, 'skills': ['Python', 'SQL']}
<class 'dict'>
Name: Asha


In [5]:
# --- Medium example: nested JSON (list of dicts) + updating + re-saving ---
students = [
    {"name": "Asha", "marks": 88, "passed": True},
    {"name": "Bibek", "marks": 40, "passed": False},
    {"name": "Chandra", "marks": 91, "passed": True},
]

with open("students.json", "w") as f:
    json.dump(students, f, indent=2)

# Read it back, update Bibek's marks (say he got remarked), and save again
with open("students.json", "r") as f:
    data = json.load(f)

for student in data:
    if student["name"] == "Bibek":
        student["marks"] = 65
        student["passed"] = student["marks"] >= 50   # recompute pass/fail

with open("students.json", "w") as f:
    json.dump(data, f, indent=2)

with open("students.json", "r") as f:
    print(f.read())


[
  {
    "name": "Asha",
    "marks": 88,
    "passed": true
  },
  {
    "name": "Bibek",
    "marks": 65,
    "passed": true
  },
  {
    "name": "Chandra",
    "marks": 91,
    "passed": true
  }
]


### ✅ Section 1 recap

- `open(path, mode)` — modes `r`, `w`, `a`, `r+` behave very differently, especially around overwriting.
- `read()` / `readline()` / `readlines()` give you the whole file, one line, or a list of lines.
- **Always prefer `with open(...) as f:`** — it closes the file automatically, even on error.
- `csv` module: `reader`/`writer` for lists, `DictReader`/`DictWriter` for dictionaries.
- `json` module: `dump`/`load` for files, `dumps`/`loads` for strings.

---


## 2. Error and Exception Handling

So far, if something went wrong (a missing file, bad data), our program
would just **crash** with a traceback. In real applications we want to
_anticipate_ likely failures and handle them gracefully.

### 2.1 The basics: `try` / `except`


In [6]:
# Simple example: division by zero
try:
    result = 10 / 0          # this line raises ZeroDivisionError
    print(result)             # never reached
except ZeroDivisionError:
    print("You can't divide by zero!")   # this runs instead of crashing

print("Program continues normally after the error was handled.")


You can't divide by zero!
Program continues normally after the error was handled.


### 2.2 The full picture: `try` / `except` / `else` / `finally`

```python
try:
    # risky code
except SomeError:
    # runs ONLY if that specific error happened
else:
    # runs ONLY if NO exception happened in the try block
finally:
    # ALWAYS runs, no matter what (error or not) - great for cleanup
```


In [7]:
# Full example combining all four blocks
def read_config(path):
    try:
        f = open(path, "r")
        content = f.read()
    except FileNotFoundError:
        print(f"[except] '{path}' not found - using default config.")
        content = None
    else:
        print("[else] File read successfully, no errors occurred.")
    finally:
        print("[finally] This always runs (e.g. good place to close resources).")
        try:
            f.close()
        except NameError:
            pass   # f was never created because open() failed
    return content

print("--- Calling with a file that EXISTS ---")
read_config("notes.txt")

print("\n--- Calling with a file that DOES NOT EXIST ---")
read_config("does_not_exist.txt")


--- Calling with a file that EXISTS ---
[else] File read successfully, no errors occurred.
[finally] This always runs (e.g. good place to close resources).

--- Calling with a file that DOES NOT EXIST ---
[except] 'does_not_exist.txt' not found - using default config.
[finally] This always runs (e.g. good place to close resources).


### 2.3 Common exceptions you will meet constantly

| Exception           | Typical cause                               |
| ------------------- | ------------------------------------------- |
| `ValueError`        | Right type, wrong value (e.g. `int("abc")`) |
| `KeyError`          | Missing dictionary key                      |
| `TypeError`         | Operation on an incompatible type           |
| `FileNotFoundError` | Trying to open a file that doesn't exist    |

Let's trigger each one on purpose, inside a `try/except`, so we can see
exactly what they look like.


In [8]:
# ValueError
try:
    age = int("twenty-one")   # "twenty-one" is not a valid number string
except ValueError as e:
    print("Caught ValueError ->", e)


Caught ValueError -> invalid literal for int() with base 10: 'twenty-one'


In [9]:
# KeyError
student = {"name": "Asha", "marks": 88}
try:
    print(student["age"])      # "age" key does not exist in this dict
except KeyError as e:
    print("Caught KeyError -> missing key:", e)


Caught KeyError -> missing key: 'age'


In [10]:
# TypeError
try:
    total = "score: " + 88     # can't concatenate str and int directly
except TypeError as e:
    print("Caught TypeError ->", e)


Caught TypeError -> can only concatenate str (not "int") to str


In [11]:
# FileNotFoundError
try:
    with open("missing_file.txt", "r") as f:
        data = f.read()
except FileNotFoundError as e:
    print("Caught FileNotFoundError ->", e)


Caught FileNotFoundError -> [Errno 2] No such file or directory: 'missing_file.txt'


### 2.4 Catching multiple exceptions

Two common patterns:

1. One `except` block for **several exception types at once** — use this
   when you want to handle them the _same_ way.
2. **Multiple `except` blocks** — use this when each error needs
   _different_ handling. Always put more **specific** exceptions before
   more **generic** ones (e.g. `Exception` last).


In [12]:
# Pattern 1: one except block, tuple of exception types
def to_number(value):
    try:
        return int(value) / 1
    except (ValueError, TypeError) as e:
        print(f"Could not convert {value!r} to a number: {e}")
        return None

to_number("abc")     # ValueError
to_number(None)       # TypeError
to_number("42")       # works fine, no exception


Could not convert 'abc' to a number: invalid literal for int() with base 10: 'abc'
Could not convert None to a number: int() argument must be a string, a bytes-like object or a real number, not 'NoneType'


42.0

In [13]:
# Pattern 2: multiple except blocks, specific -> generic
def safe_lookup(data, key):
    try:
        value = data[key]
        return 100 / value
    except KeyError:
        print(f"Key '{key}' not found in dictionary.")
    except ZeroDivisionError:
        print(f"Value for '{key}' was zero - can't divide.")
    except Exception as e:
        # generic fallback: catches anything we didn't specifically plan for
        print(f"Unexpected error: {type(e).__name__} - {e}")

data = {"a": 5, "b": 0}
safe_lookup(data, "a")   # normal case
safe_lookup(data, "b")   # ZeroDivisionError case
safe_lookup(data, "c")   # KeyError case


Value for 'b' was zero - can't divide.
Key 'c' not found in dictionary.


### 2.5 `raise` and Custom Exceptions

Sometimes _you_ are the one who knows something is wrong (e.g. invalid
business rule) — Python didn't raise an error, but it should. That's what
`raise` is for. You can also design your **own exception classes** for
domain-specific errors, which makes your code's intent much clearer than
using a generic `ValueError` everywhere.


In [14]:
# Simple: raising a built-in exception manually
def validate_age(age):
    if age < 0:
        raise ValueError(f"Age cannot be negative, got {age}")
    return age

try:
    validate_age(-5)
except ValueError as e:
    print("Validation failed ->", e)


Validation failed -> Age cannot be negative, got -5


In [15]:
# Medium: a CUSTOM exception class for a bank withdrawal system

class InsufficientBalanceError(Exception):
    # Raised when a withdrawal amount exceeds the available balance.
    pass   # we don't need extra logic - inheriting from Exception is enough

def withdraw(balance, amount):
    if amount > balance:
        # raise OUR custom, meaningful exception instead of a generic one
        raise InsufficientBalanceError(
            f"Cannot withdraw {amount}: balance is only {balance}"
        )
    return balance - amount

try:
    new_balance = withdraw(balance=1000, amount=1500)
    print("New balance:", new_balance)
except InsufficientBalanceError as e:
    print("Transaction blocked ->", e)

# Custom exceptions can still be caught as Exception too, since they inherit from it
try:
    withdraw(100, 500)
except Exception as e:
    print(f"Generic catch also works: {type(e).__name__}: {e}")


Transaction blocked -> Cannot withdraw 1500: balance is only 1000
Generic catch also works: InsufficientBalanceError: Cannot withdraw 500: balance is only 100


### 2.6 Combining File Handling + Error Handling (medium)

In real pipelines, "read a file" and "parse its contents" are two separate
failure points. Let's write one robust function that handles both a
missing file **and** malformed JSON.


In [16]:
import json

def safe_read_json(path):
    # Read a JSON file safely, returning None if something goes wrong.
    try:
        with open(path, "r") as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"[safe_read_json] File not found: {path}")
    except json.JSONDecodeError as e:
        print(f"[safe_read_json] File exists but is not valid JSON: {e}")
    except Exception as e:
        print(f"[safe_read_json] Unexpected error: {type(e).__name__}: {e}")
    return None

# Case 1: file does not exist
print("Result:", safe_read_json("no_such_file.json"))

# Case 2: file exists but contains BROKEN json (missing closing brace)
with open("broken.json", "w") as f:
    f.write('{"name": "Asha", "age": 21')   # intentionally malformed

print("Result:", safe_read_json("broken.json"))

# Case 3: a perfectly valid file (created earlier in Section 1)
print("Result:", safe_read_json("person.json"))


[safe_read_json] File not found: no_such_file.json
Result: None
[safe_read_json] File exists but is not valid JSON: Expecting ',' delimiter: line 1 column 27 (char 26)
Result: None
Result: {'name': 'Asha', 'age': 21, 'is_student': True, 'skills': ['Python', 'SQL']}


### ✅ Section 2 recap

- `try/except` prevents crashes; `else` runs on success, `finally` always runs (cleanup).
- Know the "big four": `ValueError`, `KeyError`, `TypeError`, `FileNotFoundError`.
- Catch several exceptions with a tuple, or with multiple `except` blocks (specific ➜ generic).
- Use `raise` to signal your own error conditions; define **custom exception classes** for clearer, domain-specific errors.

---


## 3. Logging Basics

### 3.1 Why logging instead of `print()`?

`print()` is fine for quick debugging, but in a real pipeline/application
you need:

- **Severity levels** — tell "just FYI" apart from "something is broken" (`print` can't do this)
- **Timestamps** — know _when_ something happened
- **Persistence** — write to a file so you can inspect what happened after a crash, without a terminal open
- **On/off control** — turn DEBUG logs off in production without deleting code
- **Consistent format** — every log line looks the same (great for searching/monitoring tools)

The built-in `logging` module gives you all of this for free.


In [2]:
import logging

# Logging LEVELS, in increasing order of severity:
#   DEBUG    (10) - detailed info, useful only when diagnosing problems
#   INFO     (20) - confirmation that things are working as expected
#   WARNING  (30) - something unexpected happened, but the program still works
#   ERROR    (40) - a serious problem, something failed
#   CRITICAL (50) - a very serious error, the program may not be able to continue

logging.basicConfig(level=logging.DEBUG)   # show everything from DEBUG upward

logging.debug("This is a DEBUG message - detailed diagnostic info")
logging.info("This is an INFO message - normal operation")
logging.warning("This is a WARNING message - something looks off")
logging.error("This is an ERROR message - something failed")
logging.critical("This is a CRITICAL message - major failure")


DEBUG:root:This is a DEBUG message - detailed diagnostic info
INFO:root:This is an INFO message - normal operation
ERROR:root:This is an ERROR message - something failed
CRITICAL:root:This is a CRITICAL message - major failure


**Note for this notebook:** because Jupyter keeps the Python process alive
between cells, `logging.basicConfig()` only takes effect the **first** time
it's called in a session. If you want to change the configuration below,
restart the kernel first (this is a Jupyter quirk, not a general Python
problem — in a normal script it "just works").

### 3.2 Custom format + logging to console


In [3]:
import logging
import importlib

# reload logging config cleanly inside a notebook (not needed in a normal .py script)
importlib.reload(logging)

logging.basicConfig(
    level=logging.INFO,                                   # minimum level to actually show
    format="%(asctime)s | %(levelname)-8s | %(message)s",   # timestamp | LEVEL | message
    datefmt="%Y-%m-%d %H:%M:%S",
)

logging.info("Pipeline started")
logging.warning("Low disk space detected")
logging.error("Failed to connect to database")


2026-08-10 19:56:20 | INFO     | Pipeline started
2026-08-10 19:56:20 | WARNING  | Low disk space detected
2026-08-10 19:56:20 | ERROR    | Failed to connect to database


### 3.3 Logging to a FILE (not just the console)

In production, logs almost always go to a **file** (or a log-management
system) so they persist after the program exits. `basicConfig(filename=...)`
sends everything to a file instead of the console.


In [17]:
import logging
import importlib

importlib.reload(logging)   # notebook-only reset, see note above

logging.basicConfig(
    filename="pipeline.log",
    filemode="w",                                          # 'w' = fresh log each run, use 'a' to keep history
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
)

logging.info("Pipeline started")
logging.warning("Retrying step 2 after timeout")
logging.error("Step 3 failed permanently")

print("Done - nothing printed to console because logs went to pipeline.log")

# Let's prove it by reading the log file back
with open("pipeline.log", "r") as f:
    print("\n--- Contents of pipeline.log ---")
    print(f.read())


Done - nothing printed to console because logs went to pipeline.log

--- Contents of pipeline.log ---
2026-08-15 18:09:49,283 | INFO     | Pipeline started
2026-08-15 18:09:49,283 | WARNING  | Retrying step 2 after timeout
2026-08-15 18:09:49,283 | ERROR    | Step 3 failed permanently



### 3.4 Medium example: a named logger with BOTH console and file handlers

Real projects avoid the "root logger" (`logging.info(...)` directly) and
instead create a **named logger** per module, with separate **handlers**
for console and file — each can even have its own level and format.


In [18]:
import logging

# 1. Create (or get) a logger with a specific name - best practice is __name__
logger = logging.getLogger("zara_pipeline")
logger.setLevel(logging.DEBUG)          # overall minimum level for this logger
logger.handlers.clear()                  # notebook-only: avoid duplicate handlers on re-run

# 2. Console handler - only show INFO and above on screen (less noisy)
console_handler = logging.StreamHandler()
console_handler.setLevel(logging.INFO)
console_format = logging.Formatter("%(levelname)-8s | %(message)s")
console_handler.setFormatter(console_format)

# 3. File handler - capture EVERYTHING (including DEBUG) for later investigation
file_handler = logging.FileHandler("zara_pipeline.log", mode="w")
file_handler.setLevel(logging.DEBUG)
file_format = logging.Formatter("%(asctime)s | %(levelname)-8s | %(name)s | %(message)s")
file_handler.setFormatter(file_format)

# 4. Attach both handlers to the logger
logger.addHandler(console_handler)
logger.addHandler(file_handler)

# --- Now use it ---
logger.debug("Loaded 3 config values")     # goes to FILE only (console min level is INFO)
logger.info("Connected to data source")     # goes to BOTH
logger.warning("2 rows had missing values") # goes to BOTH
logger.error("Failed to write output file") # goes to BOTH

print("\n--- zara_pipeline.log contents ---")
with open("zara_pipeline.log", "r") as f:
    print(f.read())


INFO     | Connected to data source
WARNING  | 2 rows had missing values
ERROR    | Failed to write output file



--- zara_pipeline.log contents ---
2026-08-15 18:14:42,134 | DEBUG    | zara_pipeline | Loaded 3 config values
2026-08-15 18:14:42,134 | INFO     | zara_pipeline | Connected to data source
2026-08-15 18:14:42,135 | WARNING  | zara_pipeline | 2 rows had missing values
2026-08-15 18:14:42,136 | ERROR    | zara_pipeline | Failed to write output file



### ✅ Section 3 recap

- `print()` is for quick checks; `logging` is for real applications.
- Levels (low ➜ high severity): `DEBUG` < `INFO` < `WARNING` < `ERROR` < `CRITICAL`.
- `basicConfig()` is the quick way to configure the root logger (console or file).
- For real projects: use `logging.getLogger(name)` + separate `StreamHandler` (console) and `FileHandler` (file), each with its own level/format.

---


## 4. Mini Project — Putting it all together

**Goal:** Read employee data from a CSV file, validate each row, log every
step (info for success, error for problems), and save the cleaned results
to a JSON file — using everything from all three sections.


In [19]:
import csv

# Step 1: create a sample CSV with one intentionally BAD row (non-numeric salary)
with open("employees.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["name", "department", "salary"])
    writer.writerow(["Asha", "Engineering", "75000"])
    writer.writerow(["Bibek", "Sales", "not_a_number"])   # bad data on purpose
    writer.writerow(["Chandra", "Engineering", "82000"])

print("employees.csv created (with one intentionally bad row).")


employees.csv created (with one intentionally bad row).


In [19]:
import csv
import json
import logging

# Step 2: set up a dedicated logger for this project
error_logger = logging.getLogger("errors_only_pipeline")
error_logger.setLevel(logging.ERROR)
error_logger.handlers.clear()

logger = logging.getLogger("employee_pipeline")
logger.setLevel(logging.DEBUG)
logger.handlers.clear()

file_handler = logging.FileHandler("employee_pipeline.log", mode="w")
file_handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-8s | %(message)s"))
console_handler = logging.StreamHandler()
console_handler.setFormatter(logging.Formatter("%(levelname)-8s | %(message)s"))

error_file_handler = logging.FileHandler("errors_only.log", mode="w")
error_file_handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-8s | %(message)s"))

logger.addHandler(file_handler)
logger.addHandler(console_handler)

error_logger.addHandler(error_file_handler)

def process_employees(csv_path, json_path):
    # Read employees from csv_path, validate each row, log progress/problems,
    # and write the CLEAN records to json_path.
    # Uses: with-open, csv.DictReader, try/except/else/finally, custom logging.
    clean_records = []
    rejected_rows = []

    try:
        with open(csv_path, "r", newline="") as f:
            reader = csv.DictReader(f)
            for row_number, row in enumerate(reader, start=1):
                try:
                    # salary must be convertible to an integer -> ValueError if not
                    salary = int(row["salary"])
                except ValueError:
                    logger.error(f"Row {row_number}: invalid salary '{row['salary']}' for {row['name']} - SKIPPED")
                    error_logger.error(f"Row {row_number}: invalid salary '{row['salary']}' for {row['name']} - SKIPPED")

                    rejected_rows.append(row)
                    continue   # skip this bad row, keep processing the rest
                except KeyError as e:
                    logger.error(f"Row {row_number}: missing expected column {e} - SKIPPED")
                    error_logger.error(f"Row {row_number}: missing expected column {e} - SKIPPED")
                    rejected_rows.append(row)
                    continue
                else:
                    # only runs if the try block succeeded (no exception)
                    clean_records.append({
                        "name": row["name"],
                        "department": row["department"],
                        "salary": salary,
                    })
                    logger.info(f"Row {row_number}: {row['name']} processed successfully")

    except FileNotFoundError:
        logger.critical(f"Input file not found: {csv_path} - aborting pipeline")
        return   # nothing more we can do

    finally:
        logger.info("Finished reading input CSV (cleanup step, always runs)")

    # Step 3: save the cleaned results as JSON
    with open(json_path, "w") as f:
        json.dump(clean_records, f, indent=2)

    logger.info(f"Saved {len(clean_records)} clean record(s) to {json_path}")

    print("\nFinal rejected data:", rejected_rows)
    
    with open("rejected_rows.json","w") as f:
        json.dump(rejected_rows,f,indent=2)

    return clean_records


result = process_employees("employees.csv", "employees_clean.json")
print("\nFinal clean records:", result)



INFO     | Row 1: Asha processed successfully
ERROR    | Row 2: invalid salary 'not_a_number' for Bibek - SKIPPED
INFO     | Row 3: Chandra processed successfully
INFO     | Finished reading input CSV (cleanup step, always runs)
INFO     | Saved 2 clean record(s) to employees_clean.json



Final rejected data: [{'name': 'Bibek', 'department': 'Sales', 'salary': 'not_a_number'}]

Final clean records: [{'name': 'Asha', 'department': 'Engineering', 'salary': 75000}, {'name': 'Chandra', 'department': 'Engineering', 'salary': 82000}]


In [20]:
# Step 4: inspect the outputs produced by the pipeline

print("--- employees_clean.json ---")
with open("employees_clean.json", "r") as f:
    print(f.read())

print("\n--- employee_pipeline.log ---")
with open("employee_pipeline.log", "r") as f:
    print(f.read())

print("\n--- errors_only.log ---")
with open("errors_only.log", "r") as f:
    print(f.read())


--- employees_clean.json ---
[
  {
    "name": "Asha",
    "department": "Engineering",
    "salary": 75000
  },
  {
    "name": "Chandra",
    "department": "Engineering",
    "salary": 82000
  }
]

--- employee_pipeline.log ---
2026-08-16 16:21:00,576 | INFO     | Row 1: Asha processed successfully
2026-08-16 16:21:00,578 | ERROR    | Row 2: invalid salary 'not_a_number' for Bibek - SKIPPED
2026-08-16 16:21:00,579 | INFO     | Row 3: Chandra processed successfully
2026-08-16 16:21:00,580 | INFO     | Finished reading input CSV (cleanup step, always runs)
2026-08-16 16:21:00,582 | INFO     | Saved 2 clean record(s) to employees_clean.json


--- errors_only.log ---
2026-08-16 16:21:00,579 | ERROR    | Row 2: invalid salary 'not_a_number' for Bibek - SKIPPED



## 🎯 Summary Cheat-Sheet

| Topic               | Key takeaway                                                                               |
| ------------------- | ------------------------------------------------------------------------------------------ |
| File modes          | `r` read, `w` overwrite, `a` append, `r+` read+write                                       |
| Reading             | `read()` whole file, `readline()` one line, `readlines()` list of lines                    |
| Best practice       | Always use `with open(...) as f:`                                                          |
| CSV                 | `csv.reader`/`writer` (lists), `DictReader`/`DictWriter` (dicts)                           |
| JSON                | `dump`/`load` (files), `dumps`/`loads` (strings)                                           |
| Errors              | `try/except/else/finally`; know `ValueError`, `KeyError`, `TypeError`, `FileNotFoundError` |
| Multiple exceptions | tuple in one `except`, or several `except` blocks (specific ➜ generic)                     |
| Custom errors       | subclass `Exception`, `raise YourError("message")`                                         |
| Logging             | `DEBUG < INFO < WARNING < ERROR < CRITICAL`; use `getLogger` + handlers for console/file   |

### 📝 Practice exercises (for after class)

1. Write a function that reads any CSV and reports, for each column, how many values were missing/blank.
2. Create a custom exception `NegativeQuantityError` and use it inside an inventory `remove_stock(qty)` function.
3. Extend the Mini Project so **all** skipped rows are also written to a separate `rejected_rows.json`, not just logged.
4. Configure the pipeline logger so ERROR and above also get written to a _second_, separate file called `errors_only.log`.


In [11]:
import csv

fieldnames =["name","age","class"]

with open("students_empty.csv","w",newline="") as f:
    write = csv.DictWriter(f,fieldnames=fieldnames)
    write.writeheader()

In [12]:
# 1. Write a function that reads any CSV and reports, for each column, how many values were missing/blank
import csv
import logging

def csv_reader(csv_file,log_file):

    logger = logging.getLogger("filereader")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()

    file_handler = logging.FileHandler(log_file,mode="w")
    file_formatter = logging.Formatter("%(asctime)s | %(levelname)-8s| %(message)s")
    file_handler.setFormatter(file_formatter)

    logger.addHandler(file_handler)

    try:
        with open(csv_file,"r",newline="") as f:
            logger.info("reading the file has been started")

            reader = csv.DictReader(f)
            missing = {column: 0 for column in reader.fieldnames}
            print(missing)
            for row in reader:
                for colunm in reader.fieldnames:
                    if row[colunm] == "":
                        missing[colunm] += 1

            for column, value in missing.items():
                print(f"{column}: {value}")

    except FileNotFoundError:
        logger.error("Failed file not found")


csv_reader("students_empty.csv","reader.log")
        
    

{'name': 0, 'age': 0, 'class': 0}
name: 1
age: 1
class: 1


In [14]:
# for inventory

inventory = [
    {
       "name":"Bread",
       "qty":-1,
       "price":130.0
    }
]

class NegativeQuantityError(Exception):
    pass

try:
    for index,inv in enumerate(inventory,start=1):
        if inv["qty"] < 0:
            raise NegativeQuantityError(f"ID: {index} Quantity is negative for {inv["name"]}")
except NegativeQuantityError as e:
    print(f"NegativeQuantityError: {e}")


NegativeQuantityError: ID: 1 Quantity is negative for Bread
